In [1]:
from sklearn.datasets import load_wine
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO


In [2]:

# Load dataset
wine = load_wine()
data = pd.DataFrame(wine.data, columns=wine.feature_names)
data['target'] = wine.target


In [3]:
# Check dataset
print(data.head())
print(data.dtypes)

# Stratified split based on target
feature_cols = [col for col in data.columns if col != 'target']


   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  target  
0          

In [4]:
# prompt: strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
# train_idx, test_idx = next(strat_shuff_split.split(data[feature_cols], data['target'])) ...generate the meaning of it

# Meaning of the code:
# strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
# This line creates an instance of the StratifiedShuffleSplit class.
# - n_splits=1: Specifies that the splitting process should be performed only once.
# - test_size=0.3: Indicates that the test set should constitute 30% of the data.
# - random_state=42: Sets a seed for the random number generator, ensuring that the split is reproducible.

# train_idx, test_idx = next(strat_shuff_split.split(data[feature_cols], data['target']))
# This line performs the actual splitting of the data.
# - strat_shuff_split.split(data[feature_cols], data['target']): This method generates indices for the training and testing sets. It takes the features (data[feature_cols]) and the target variable (data['target']) as input to ensure that the splitting is stratified based on the target variable's distribution.
# - next(...): Since n_splits=1, the `split` method yields a single pair of indices (for the training and testing sets). The `next()` function is used to retrieve this single pair.
# - train_idx, test_idx = ...: The retrieved pair of indices is then unpacked into two variables: `train_idx` (indices for the training set) and `test_idx` (indices for the testing set).

# In summary, these two lines create a single split of the dataset into training (70%) and testing (30%) sets, ensuring that the proportion of each class in the target variable is maintained in both sets, making the split suitable for classification tasks where class distribution is important.


In [5]:
strat_shuff_split = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(strat_shuff_split.split(data[feature_cols], data['target']))


In [6]:
X_train = data.loc[train_idx, feature_cols]
y_train = data.loc[train_idx, 'target']
X_test = data.loc[test_idx, feature_cols]
y_test = data.loc[test_idx, 'target']


In [7]:
print("\nTarget distribution in train:")
print(y_train.value_counts(normalize=True).sort_index())
print("\nTarget distribution in test:")
print(y_test.value_counts(normalize=True).sort_index())

# Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("\nTree depth:", dt.tree_.max_depth)
print("Node count:", dt.tree_.node_count)



Target distribution in train:
target
0    0.330645
1    0.403226
2    0.266129
Name: proportion, dtype: float64

Target distribution in test:
target
0    0.333333
1    0.388889
2    0.277778
Name: proportion, dtype: float64

Tree depth: 4
Node count: 15


In [8]:
def measure_error(y_true, y_pred, label):
    return pd.Series({
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1': f1_score(y_true, y_pred, average='weighted')
    }, name=label)

y_train_pred = dt.predict(X_train)
y_test_pred = dt.predict(X_test)
print("\nDecision Tree Performance:")
print(pd.concat([
    measure_error(y_train, y_train_pred, 'train'),
    measure_error(y_test, y_test_pred, 'test')
], axis=1))



Decision Tree Performance:
           train      test
accuracy     1.0  0.962963
precision    1.0  0.966184
recall       1.0  0.962963
f1           1.0  0.963221


In [9]:
# Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)


In [10]:
print("\nRandom Forest Feature Importances:")
print(dict(zip(X_train.columns, rf.feature_importances_)))

print("\nRandom Forest Performance:")
print(pd.concat([
    measure_error(y_train, y_train_pred_rf, 'train'),
    measure_error(y_test, y_test_pred_rf, 'test')
], axis=1))



Random Forest Feature Importances:
{'alcohol': np.float64(0.16271453707343966), 'malic_acid': np.float64(0.033700116664420826), 'ash': np.float64(0.016418427375599065), 'alcalinity_of_ash': np.float64(0.028840691920682088), 'magnesium': np.float64(0.03459085050934072), 'total_phenols': np.float64(0.04200407614082344), 'flavanoids': np.float64(0.15628344294168045), 'nonflavanoid_phenols': np.float64(0.011825103487317289), 'proanthocyanins': np.float64(0.019951941321779146), 'color_intensity': np.float64(0.15971900668395872), 'hue': np.float64(0.11101871490496351), 'od280/od315_of_diluted_wines': np.float64(0.09885162944276034), 'proline': np.float64(0.12408146153323467)}

Random Forest Performance:
           train  test
accuracy     1.0   1.0
precision    1.0   1.0
recall       1.0   1.0
f1           1.0   1.0


In [11]:
# Grid Search with Decision Tree
param_grid = {
    'max_depth': range(1, dt.get_depth() + 1, 2),
    'max_features': range(1, len(X_train.columns) + 1)
}
GR = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, scoring='accuracy', n_jobs=-1)
GR.fit(X_train, y_train)

print("\nBest Tree from GridSearch - Depth:", GR.best_estimator_.tree_.max_depth)
print("Best Tree Node Count:", GR.best_estimator_.tree_.node_count)

y_train_pred_gr = GR.predict(X_train)
y_test_pred_gr = GR.predict(X_test)
print("\nGridSearch Tree Performance:")
print(pd.concat([
    measure_error(y_train, y_train_pred_gr, 'train'),
    measure_error(y_test, y_test_pred_gr, 'test')
], axis=1))



Best Tree from GridSearch - Depth: 3
Best Tree Node Count: 13

GridSearch Tree Performance:
              train      test
accuracy   0.975806  0.888889
precision  0.976349  0.893160
recall     0.975806  0.888889
f1         0.975740  0.887843
